# Case Method — Segmentasi Produk E-Commerce (K-Medoids)
## Tahap 2 & 3: Pemahaman Dataset & Data Preprocessing

Notebook ini mengubah data transaksi mentah (`Online Retail.xlsx` dari UCI) menjadi
tabel **RFM per produk** (Recency, Frequency, Quantity) yang siap dipakai sebagai
input algoritma K-Medoids.

**Cara pakai:**
1. Jalankan Cell 1 (setup folder) satu kali.
2. Jalankan Cell 2, lalu upload file `Online Retail.xlsx` saat diminta.
3. Jalankan sisa cell dari atas ke bawah secara berurutan.


### Cell 1 — Setup struktur folder

In [ ]:
import os

os.makedirs("case_method_kmedoids/data/extracted", exist_ok=True)
os.makedirs("case_method_kmedoids/output", exist_ok=True)
os.chdir("case_method_kmedoids")

print("Struktur folder siap. Direktori kerja sekarang:", os.getcwd())


### Cell 2 — Upload dataset mentah (Online Retail.xlsx)

In [ ]:
from google.colab import files

print("Pilih file 'Online Retail.xlsx' yang sudah kamu download dari UCI:")
uploaded = files.upload()

nama_file = list(uploaded.keys())[0]
os.rename(nama_file, f"data/extracted/{nama_file}")
RAW_XLSX = f"data/extracted/{nama_file}"
print(f"\nFile tersimpan di: {RAW_XLSX}")


---
## TAHAP 2 — Pemahaman Dataset

Membaca file Excel mentah, menyimpan salinan CSV (lebih cepat dibaca ulang),
lalu memeriksa kualitas datanya: dimensi, missing value, duplikat, dan
indikasi transaksi tidak valid. Tahap ini **tidak mengubah data apa pun**,
murni untuk pengambilan keputusan cleaning di tahap berikutnya.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RAW_CSV = "data/online_retail_raw.csv"

df = pd.read_excel(RAW_XLSX, engine="openpyxl")
df.to_csv(RAW_CSV, index=False)

print("=" * 70)
print("TAHAP 2 - PEMERIKSAAN KUALITAS DATA MENTAH")
print("=" * 70)
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom : {df.shape[1]}")

missing = df.isnull().sum()
print("\nMissing value per kolom:")
for col, n in missing[missing > 0].items():
    print(f"  {col:15s}: {n:,} ({n/len(df)*100:.2f}%)")

n_dup = df.duplicated().sum()
print(f"\nBaris duplikat persis: {n_dup:,} ({n_dup/len(df)*100:.2f}%)")

n_cancel = df["InvoiceNo"].astype(str).str.startswith("C").sum()
n_qty_neg = (df["Quantity"] <= 0).sum()
n_price_neg = (df["UnitPrice"] <= 0).sum()
print(f"Transaksi pembatalan (InvoiceNo awalan 'C'): {n_cancel:,}")
print(f"Baris Quantity <= 0  : {n_qty_neg:,}")
print(f"Baris UnitPrice <= 0 : {n_price_neg:,}")
print(f"Produk unik (StockCode) sebelum cleaning: {df['StockCode'].nunique():,}")

df.head()


---
## TAHAP 3a — Cleaning Level Baris Transaksi (umum)

Langkah ini berlaku untuk seluruh dataset (dipakai baik untuk analisis
per pelanggan maupun per produk):

1. Isi `Description` kosong dengan `'UNKNOWN'` (tidak menghapus baris — kolom
   ini hanya label, tidak dipakai di perhitungan RFM).
2. Hapus transaksi pembatalan (`InvoiceNo` berawalan `'C'`).
3. Hapus baris `Quantity <= 0` (sisa retur yang tidak tertangkap langkah 2).
4. Hapus baris duplikat persis.
5. Hapus baris `UnitPrice <= 0` (bukan transaksi penjualan valid).

Catatan: baris tanpa `CustomerID` **tidak** dihapus di sini, karena untuk
analisis per produk, identitas pelanggan tidak dibutuhkan.


In [ ]:
print("=" * 70)
print("TAHAP 3a - CLEANING LEVEL BARIS TRANSAKSI")
print("=" * 70)
n0 = len(df)

df["Description"] = df["Description"].fillna("UNKNOWN")

before = len(df)
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
print(f"Hapus transaksi pembatalan : {before-len(df):,} baris -> sisa {len(df):,}")

before = len(df)
df = df[df["Quantity"] > 0]
print(f"Hapus Quantity <= 0        : {before-len(df):,} baris -> sisa {len(df):,}")

before = len(df)
df = df.drop_duplicates()
print(f"Hapus duplikat persis      : {before-len(df):,} baris -> sisa {len(df):,}")

before = len(df)
df = df[df["UnitPrice"] > 0]
print(f"Hapus UnitPrice <= 0       : {before-len(df):,} baris -> sisa {len(df):,}")

print(f"\nTotal baris dihapus: {n0-len(df):,} ({(n0-len(df))/n0*100:.2f}%)")
print(f"Baris tersisa: {len(df):,}")

df_clean = df.copy()
df_clean.to_csv("data/online_retail_clean.csv", index=False)


---
## TAHAP 3b — Cleaning & Agregasi Khusus Level Produk

Langkah tambahan khusus untuk analisis per **produk**:

1. **Buang kode non-produk** — StockCode yang sebenarnya biaya administratif
   (ongkir, biaya bank, koreksi manual, dll), bukan barang fisik.
2. **Satukan nama produk** — satu StockCode kadang punya beberapa variasi
   penulisan Description; dipilih yang paling sering muncul (modus).
3. **Agregasi ke level produk** — hitung atribut Recency, Frequency, Quantity
   per StockCode. Inilah **input utama algoritma K-Medoids**.


In [ ]:
print("=" * 70)
print("TAHAP 3b - MEMBUANG KODE NON-PRODUK")
print("=" * 70)

NON_PRODUCT_CODES = ["POST", "DOT", "M", "BANK CHARGES", "PADS", "CRUK", "C2", "S", "AMAZONFEE"]
is_non_product = df_clean["StockCode"].astype(str).str.upper().isin(NON_PRODUCT_CODES)
removed = df_clean.loc[is_non_product].groupby("StockCode")["Description"].first()
for code_, desc in removed.items():
    n_rows = (df_clean["StockCode"] == code_).sum()
    print(f"  Dibuang: {code_:15s} ({desc}) -- {n_rows:,} baris")

before = len(df_clean)
df_product = df_clean[~is_non_product].copy()
print(f"\nTotal baris dibuang: {before-len(df_product):,} -> sisa {len(df_product):,}")
print(f"Produk unik setelah dibersihkan: {df_product['StockCode'].nunique():,}")

df_product.to_csv("data/online_retail_clean_product.csv", index=False)


In [ ]:
print("=" * 70)
print("MENYATUKAN NAMA PRODUK (MODUS PER STOCKCODE)")
print("=" * 70)

desc_variation = df_product.groupby("StockCode")["Description"].nunique()
n_multi = (desc_variation > 1).sum()
print(f"StockCode dengan >1 variasi nama: {n_multi:,} dari {len(desc_variation):,} produk")

mode_desc = (
    df_product.groupby("StockCode")["Description"]
    .agg(lambda x: x.value_counts().idxmax())
    .rename("ProductName")
)

print("\nContoh hasil pemilihan nama representatif:")
for code_ in desc_variation[desc_variation > 1].index[:3]:
    variations = df_product.loc[df_product["StockCode"] == code_, "Description"].value_counts()
    print(f"  StockCode {code_}: variasi = {dict(variations)} -> dipilih: '{mode_desc[code_]}'")


In [ ]:
print("=" * 70)
print("AGREGASI KE LEVEL PRODUK: MEMBENTUK ATRIBUT RECENCY-FREQUENCY-QUANTITY")
print("=" * 70)

df_product["InvoiceDate"] = pd.to_datetime(df_product["InvoiceDate"])
snapshot_date = df_product["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Snapshot date (acuan Recency): {snapshot_date}")

product_rfm = df_product.groupby("StockCode").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Quantity=("Quantity", "sum"),
).reset_index()

product_rfm = product_rfm.merge(mode_desc.reset_index(), on="StockCode", how="left")
product_rfm = product_rfm[["StockCode", "ProductName", "Recency", "Frequency", "Quantity"]]

print(f"Jumlah produk hasil agregasi: {len(product_rfm):,}")
print("\nStatistik deskriptif:")
print(product_rfm[["Recency", "Frequency", "Quantity"]].describe().round(2))

product_rfm.to_csv("data/product_rfm.csv", index=False)
print("\nDisimpan ke: data/product_rfm.csv")

product_rfm.head(10)


---
## Selesai — Preview Hasil Akhir & Download

Tabel `product_rfm.csv` ini adalah **input utama Tahap 4 (Implementasi K-Medoids)**.


In [ ]:
print("10 produk dengan Quantity terjual tertinggi:")
display(product_rfm.sort_values("Quantity", ascending=False).head(10))

print("\n10 produk dengan Recency terbesar (paling lama tidak terjual):")
display(product_rfm.sort_values("Recency", ascending=False).head(10))

# Download hasil akhir ke komputer kamu
files.download("data/product_rfm.csv")
